In [1]:
import pandas as pd 
import numpy as np 

## data cleaning and handling 

In [4]:
product_info= pd.read_csv("product_info.csv")

In [5]:
# og_file.tail(5)

In [6]:
import glob

In [7]:
files = glob.glob("reviews_*.csv")
len(files)

5

In [8]:
reviews=pd.concat([pd.read_csv(file) for file in files],ignore_index=True)

In [9]:
null_counts=reviews.isnull().sum()
null_counts

Unnamed: 0.1                     0
Unnamed: 0                       0
rating                           0
is_recommended               57232
helpfulness                 154322
total_feedback_count             0
total_neg_feedback_count         0
total_pos_feedback_count         0
submission_time                  0
review_text                    345
review_title                 79694
skin_tone                    55226
eye_color                    67746
skin_type                    36874
hair_color                   71081
product_id                       0
product_name                     0
brand_name                       0
price_usd                        0
dtype: int64

In [10]:
# reviews.tail()
reviews['is_recommended'].unique()

array([ 1.,  0., nan])

In [11]:

# reviews=reviews.drop(columns=['eye_color','hair_color'])
reviews=reviews.dropna(subset=['review_text'])
print(len(reviews))

reviews['helpfulness'].unique()
# helpfulness has 1349 unique values which makes no sense when 154322 values are null 

285067


array([1.        ,        nan, 0.25      , ..., 0.65853697, 0.640351  ,
       0.85981297])

In [12]:
reviews['skin_tone']=reviews['skin_tone'].fillna('unknown')
reviews['skin_type']=reviews['skin_type'].fillna('unknown')
# reviews['review_title_text']=reviews['review_title']+reviews['review_text']

In [13]:
reviews=reviews.drop(columns=["Unnamed: 0.1","Unnamed: 0"])

In [14]:
# reviews.head()
# reviews=reviews.drop(revi
# reviews[reviews['review_title'].isnull()]
# reviews=reviews.drop(columns=['review_title_text'])
# def recommedation_system(arr):
#     for i in arr :
        

In [15]:
reviews['review_title'] = reviews['review_title'].fillna('')
reviews['is_recommended']=( reviews.groupby('product_id')['is_recommended'].transform
(lambda x : x.fillna(x.mode()[0] if not x.mode().empty else 0)))


In [16]:
reviews=reviews.drop(columns=['eye_color','hair_color'])
reviews['helpfulness']=(reviews.groupby('product_id')['helpfulness'].transform(lambda x:x.fillna(round(x.mean(),2)if not x.mean() else 0)))

In [17]:
# Cell: Run this immediately after your concat line
print(type(reviews))
# print(reviews)

<class 'pandas.core.frame.DataFrame'>


In [18]:
reviews.isnull().sum()

rating                      0
is_recommended              0
helpfulness                 0
total_feedback_count        0
total_neg_feedback_count    0
total_pos_feedback_count    0
submission_time             0
review_text                 0
review_title                0
skin_tone                   0
skin_type                   0
product_id                  0
product_name                0
brand_name                  0
price_usd                   0
dtype: int64

In [19]:
# reviews.duplicated().sum()
# reviews.drop_duplicates()
# removed duplicates

## weak labeling 

In [21]:
def labelling(rate):
    if rate > 3 :
        return 'positive'
    elif rate == 3 :
        return 'neutral'
    else:
        return 'negetive'
reviews['sentiment']=reviews['rating'].apply(labelling)

print(reviews['sentiment'].value_counts(normalize=True).mul(100).round(2))

reviews['sentiment'].value_counts()
    

sentiment
positive    82.85
negetive     9.73
neutral      7.42
Name: proportion, dtype: float64


sentiment
positive    236191
negetive     27728
neutral      21148
Name: count, dtype: int64

In [22]:
cross = pd.crosstab(reviews['sentiment'], reviews['is_recommended'])
print(cross)

is_recommended    0.0     1.0
sentiment                    
negetive        22352    5376
neutral         11572    9576
positive         1697  234494


# Preprocessing and tokenization 

In [24]:
reviews['combined_review']=reviews['review_text']+" "+ reviews['review_title']
# reviews.tail()
reviews['combined_review'] = reviews['combined_review'].str.lower()
# reviews=reviews.drop(columns=['comined_review'])

In [25]:
# reviews.head()
import re

def clean_text(text):
    text = re.sub(r'http\S+', '', text)           
    text = re.sub(r'[^a-z\s]', '', text)         
    text = re.sub(r'\s+', ' ', text).strip()      
    return text

reviews['combined_review'] = reviews['combined_review'].apply(clean_text)

In [26]:
reviews.to_csv('cleaned_reviews.csv', index=False)

In [27]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
from nltk.tokenize import word_tokenize
reviews['tokens']=reviews['combined_review'].apply(word_tokenize) 

nltk.download('stopwords')
from nltk.corpus import stopwords
stop_words=set(stopwords.words('english'))
sentimental_words={'no','not','never','very','too','more', 'most'}
stop_words=stop_words-sentimental_words
reviews['tokens']=reviews['tokens'].apply(lambda tokens: [t for t in tokens if t not in stop_words])
# reviews.head()

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Varshini\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Varshini\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Varshini\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [28]:
nltk.download('wordnet')
from nltk.stem import WordNetLemmatizer
lemmatizer=WordNetLemmatizer()
reviews['tokens']= reviews['tokens'].apply(lambda tokens: [lemmatizer.lemmatize(t) for t in tokens])


[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Varshini\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [66]:
reviews=reviews.drop(columns=['total_neg_feedback_count', 'total_pos_feedback_count', 'total_feedback_count'])
# reviews.head()
reviews.columns

Index(['rating', 'is_recommended', 'helpfulness', 'submission_time',
       'review_text', 'review_title', 'skin_tone', 'skin_type', 'product_id',
       'product_name', 'brand_name', 'price_usd', 'sentiment',
       'combined_review', 'tokens'],
      dtype='object')

In [60]:
reviews.to_csv('cleaned_reviews.csv', index=False)

In [62]:
product_skincare=pd.read_csv("product_info_skincare.csv")
cleaned_revieww= pd.read_csv("cleaned_reviews.csv")
cleaned_revieww.columns

Index(['rating', 'is_recommended', 'helpfulness', 'total_feedback_count',
       'total_neg_feedback_count', 'total_pos_feedback_count',
       'submission_time', 'review_text', 'review_title', 'skin_tone',
       'skin_type', 'product_id', 'product_name', 'brand_name', 'price_usd',
       'sentiment', 'combined_review', 'tokens'],
      dtype='object')

In [80]:
product_skincare.head()
len(product_skincare)

product_skincare=product_skincare.drop(columns=['tertiary_category',
       'child_count', 'child_max_price', 'child_min_price','brand_id','child_count'
     ,'variation_desc','value_price_usd','online_only','tertiary_category','reviews'
    ,'sephora_exclusive','Unnamed: 0','limited_edition'])

product_skincare.to_csv("clean_product_skincare.csv",index=False)  
clean_skincare=pd.read_csv("clean_product_skincare.csv") 
product_skincare.columns
# product_skincare.head()

Index(['product_id', 'product_name', 'brand_name', 'loves_count', 'rating',
       'size', 'variation_type', 'variation_value', 'ingredients', 'price_usd',
       'sale_price_usd', 'new', 'out_of_stock', 'highlights',
       'primary_category', 'secondary_category'],
      dtype='object')

In [72]:
reviews=reviews.drop(columns=['brand_name','product_name'])

In [172]:
# reviews.to_csv('cleaned_reviews.csv', index=False)
cleaned_reviews= pd.read_csv("cleaned_reviews.csv")
cleaned_reviews.columns

Index(['rating', 'is_recommended', 'helpfulness', 'submission_time',
       'review_text', 'review_title', 'skin_tone', 'skin_type', 'product_id',
       'price_usd', 'sentiment', 'combined_review', 'tokens'],
      dtype='object')

In [174]:
# import seaborn as sns 
# sns.heatmap(reviews.corr(),annot=true)
master=pd.merge(clean_skincare,cleaned_reviews,on='product_id',how='left')


NameError: name 'clean_skincare' is not defined

In [3]:
# master.info()
master.to_csv('master.csv',index=False)

NameError: name 'master' is not defined

In [35]:
# master=master.drop(columns=['total_feedback_count', 'total_neg_feedback_count',
#        'total_pos_feedback_count','total_feedback_count', 'total_neg_feedback_count',
#        'total_pos_feedback_count'])

## exploratry data analysis 

In [68]:
# master.isnull().sum()

In [176]:
master.shape
cleaned_reviews.shape
products.shape

NameError: name 'products' is not defined

In [86]:
# master.to_csv('master.csv',index=False)

In [18]:
master = master.rename(columns={
    'rating_x':'product_rating',
    'rating_y':'review_rating'
})
master.columns

Index(['product_id', 'product_name', 'brand_name', 'loves_count',
       'product_rating', 'size', 'variation_type', 'variation_value',
       'ingredients', 'price_usd_x', 'sale_price_usd', 'new', 'out_of_stock',
       'highlights', 'primary_category', 'secondary_category', 'review_rating',
       'is_recommended', 'helpfulness', 'submission_time', 'review_text',
       'review_title', 'skin_tone', 'skin_type', 'price_usd_y', 'sentiment',
       'combined_review', 'tokens'],
      dtype='object')

In [7]:
master=pd.read_csv("master.csv")

In [9]:
counts_of_sentiment=master['sentiment'].value_counts()

In [11]:
counts_of_sentiment = counts_of_sentiment.to_frame().reset_index()
counts_of_sentiment.columns = ['sentiment', 'count']

In [13]:
recommedation_analysis=master['is_recommended'].value_counts(normalize=True)
recommedation_analysis.columns=['is_recommended','percent']

### skin type and brand analysis

In [20]:
skin_type_rating=master.groupby('skin_type',as_index=False)['review_rating'].mean()
skin_type_rating.columns=['skin_type','rating']
# skin_type_rating['rating']
brand_rating=master.groupby('brand_name',as_index=False)['review_rating'].mean()
brand_rating.columns=['brand_name','rating']
print(brand_rating)
print(skin_type_rating)

category_rating=master.groupby('primary_category',as_index=False)['review_rating'].mean()
category_rating.columns=['category','ratings']
print(category_rating)

            brand_name    rating
0               ABBOTT       NaN
1                AERIN       NaN
2     ALTERNA Haircare       NaN
3             Algenist  4.254167
4              Alpha-H  4.505618
..                 ...       ...
191  iNNBEAUTY PROJECT  4.505155
192          innisfree  4.589523
193         philosophy       NaN
194         shu uemura       NaN
195              tarte  4.383194

[196 rows x 2 columns]
     skin_type    rating
0  combination  4.332884
1          dry  4.319378
2       normal  4.306481
3         oily  4.318846
4      unknown  4.365804
          category   ratings
0      Bath & Body       NaN
1        Fragrance       NaN
2             Hair       NaN
3           Makeup       NaN
4              Men       NaN
5        Mini Size       NaN
6         Skincare  4.330449
7  Tools & Brushes       NaN


In [80]:
# master.isnull().sum()
# product_groups=master.groupby('product_id',as_index=False).mean()
# product_groups.head()

In [ ]:
# product_ids=product_groups['product_id'].unique()
# product_ids.count()

In [204]:
# product_groups.isnull().sum()
# avg_rating = product_groups['rating_x'].mean()
master_reviews = master[
    master['combined_review'].notna()
]
master_reviews['product_id'].nunique()


541

In [205]:
cols = [
    'product_id',
    'product_name',
    'brand_name',
    'combined_review',
    'product_rating',     
    'sentiment',
    'skin_type',
    'is_recommended',
    'ingredients',
    'highlights',
    'primary_category',
    'price_usd_x'
]

master_reviews= master_reviews[cols]

In [238]:
grouped = master_reviews.groupby('product_id')
master['product_id'].nunique()
len(grouped)
print(grouped.ngroups)
product_intelligence=[]
# master[['product_id','combined_review']].head()

541


In [234]:
for product_id,group in grouped :
    product_name=group['product_name'].iloc[0]
    brand=group['brand_name'].iloc[0]
    avg_rating=round(group['product_rating'].mean(),2)
    if group['is_recommended'].notna().sum() > 0:
        recommendation_rate = round(group['is_recommended'].mean()*100, 2)
    else:
        recommendation_rate = "Not Available"    
    ingredients = str( group['ingredients'].iloc[0]).replace("[","").replace("]","")    
    highlights=str(group['highlights'].iloc[0])
    sample_reviews="\n".join(group['combined_review'].dropna().astype(str).head(20).tolist())
    if len(sample_reviews.strip()) == 0:
        sample_reviews = "No customer reviews available."
    document = f"""
    Product Name: {product_name}

    Brand: {brand}

    Average Rating: {avg_rating}

    Recommendation Rate:
    {recommendation_rate}%

    Ingredients:
    {ingredients}

    Highlights:
    {highlights}

    Customer Reviews:
    {sample_reviews}
    """

    product_intelligence.append(document)

In [236]:
len(documents)
# documents[0]
# # print(sample_reviews[:1000])
# # documents.shape()
# import numpy as
# print(np.shape(document))
# print(group['combined_review'].head(10),
# group['combined_review'].dropna().shape,
# group['combined_review'].head())

7062

In [162]:
# first_product_id = list(grouped.groups.keys())[0]

# first_group = grouped.get_group(first_product_id)

# print(first_group['product_name'].iloc[0])
# print(first_group['combined_review'].head())

Brow Harmony Pencil & Gel
214836    NaN
Name: combined_review, dtype: object


In [164]:
master.groupby('product_id')['combined_review'].count().sort_values().head(10)

product_id
P05048798    0
P473328      0
P473264      0
P473260      0
P473155      0
P473154      0
P473153      0
P473152      0
P473146      0
P473120      0
Name: combined_review, dtype: int64

In [186]:
master['combined_review'].isna().sum()
print(master.shape,
cleaned_reviews.shape)
# grouped.shape

(286339, 12) (285067, 13)


In [192]:
# documents.to_csv('product_intelligence.csv',index=False)
# test_id = "P473328"

# print(
#     cleaned_reviews[
#         cleaned_reviews['product_id'] == test_id
#     ].shape
# )

# cleaned_reviews[
#     cleaned_reviews['product_id'] == test_id
# ][['product_name','combined_review']].head()
master['product_id'].nunique()

1813

In [108]:
pip install sentence-transformers

Note: you may need to restart the kernel to use updated packages.
   ---------------------------------------- 0.0/588.9 kB ? eta -:--:--
   ---------------------------------------- 588.9/588.9 kB 3.5 MB/s eta 0:00:00
   ---------------------------------------- 0.0/693.4 kB ? eta -:--:--
   ---------------------------------------- 693.4/693.4 kB 7.0 MB/s eta 0:00:00
   ---------------------------------------- 0.0/123.0 MB ? eta -:--:--
    --------------------------------------- 2.1/123.0 MB 10.7 MB/s eta 0:00:12
   - -------------------------------------- 3.9/123.0 MB 10.2 MB/s eta 0:00:12
   -- ------------------------------------- 6.3/123.0 MB 10.2 MB/s eta 0:00:12
   -- ------------------------------------- 8.1/123.0 MB 9.9 MB/s eta 0:00:12
   --- ------------------------------------ 10.2/123.0 MB 10.1 MB/s eta 0:00:12
   ---- ----------------------------------- 12.3/123.0 MB 10.2 MB/s eta 0:00:11
   ---- ----------------------------------- 14.4/123.0 MB 10.1 MB/s eta 0:00:11
   ---

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
streamlit 1.37.1 requires protobuf<6,>=3.20, but you have protobuf 6.33.1 which is incompatible.
streamlit 1.37.1 requires rich<14,>=10.14.0, but you have rich 15.0.0 which is incompatible.


In [114]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer(
    'all-MiniLM-L6-v2'
)
print("Model loaded successfully")
# embeddings = model.encode(
#     documents,
#     show_progress_bar=True
# )

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model loaded successfully


In [116]:
embeddings = model.encode(
    ["hello world"]
)

print(embeddings.shape)

(1, 384)
